# System Architecture and Communication Topology
---

In this notebook, we demonstrated how to explore system topology and intra-node/inter-node communication concepts. We then delved into the NVIDIA Collective Communication Library (NCCL), detailing five collective communication primitives: `AllReduce,` `Broadcast,` `Reduce,` `AllGather,` and `ReduceScatter.`    

## Important Terminology

- **Host**: The CPU and its memory (host memory)
- **Device**: The GPU and its memory (device memory)
- **Kernel**:  A GPU function executed on the device.
- **Latency:** The time it takes to take a data unit from point A to point B. For example, if 4B of data can be transferred from point A to B in 4 $\mu$s, the 4 $\mu$s is referred to as the transfer latency.
- **Bandwidth:** The amount of data that can be transferred from point A to point B in a unit of time. For example, if the width of the bus is 64KB and the latency of transfer between points A and B is 4 $\mu$s, the bandwidth is 64KB * (1/4$\mu$s) = 1.6 GB/s.


## Communication Approaches   

Several GPU communication approaches involve data copying/movement operations. Let’s look at two common basic concepts: Host Staging of Copy Operations and Peer-to-Peer Memory Access. Understanding these concepts will lay a foundation for more complex approaches to be demonstrated in the rest of this content.

### I. Host Staging of Copy Operations

The diagram below illustrates data movement (following the red arrow path) from `GPU 0` to `GPU 1` in a GPU-to-GPU memory copy operation. The data moves from `GPU 0` and passes through the PICe bus to the CPU, which is staged in a buffer before being copied to `GPU 1`. This process is called host stagging, which thus reduces bandwidth and increases latency. Performance can be improved by eliminating host staging.

<img src="images/DL_host_staging.png" width="380px" height="380px" alt-text="Host"/>


### II. Peer-to-Peer Memory Access

The P2P approach allows GPUs to address each other's memory from within device kernels. It eliminates host staging by transferring data either through the PCIe switch or through NVLink, as denoted by the red arrows in the diagram below. For example,  data can be transversed directly from `GPU 0` to `GPU 1` via NVLink or from `GPU 2` to `GPU 3` via the PCIe. This approach requires GPUs to share a Unified Virtual Address Space (UVA). UVA means that single address space is used for the host and all modern NVIDIA GPU devices (specifically, those with a compute capability of 2.0 or higher).

<img src="images/DL_p2p.png" width="380px" height="380px" alt-text="p2p"/>


Let's check if P2P is supported between the GPUs by running the command in the cell below:

In [ ]:
!srun --partition=primary -n2 --gres=gpu:8 nvidia-smi topo -p2p r

**Likely output on H100**:

<img src="images/H100-p2p-r.png" width="380px" height="380px" alt-text="p2pr"/>

This means all the GPUs can communicate via P2P with each other through PCIe (You will also find similar results in DGX A100). To check whether P2P via NVLink is supported, please run the command below:

In [ ]:
!srun --partition=primary -n1 --gres=gpu:4 nvidia-smi topo -p2p n

**Likely output on H100**:

<img src="images/H100-p2p-n.png" width="380px" height="380px" alt-text="p2pn"/>

This implies that all the GPUs can communicate with each other in P2P via NVLink. Therefore, host staging is eliminated, and performance is improved. 
The `topo` sub-command requests information on the GPU communication topology. The `-p2p` or `–p2p2status` flag displays the `p2p` status between the GPUs of a given `p2p` capability as listed below:

- **r**: p2p read capability
- **w**: p2p write capability
- **n**: p2p nvlink capability
- **a**: p2p atomic capability
- **p**: p2p prop capability 


## Intra-Node Commumications

This section will consider the DGX H100 and B200 communication architecture and the Network Interface Card (NIC) topology.



### DGXC H100 Topology

The NVIDIA H100 GPU is based on the Hopper GPU architecture and features multiple innovations, which include:

- Fourth-generation Tensor Cores that perform faster matrix computations even on a broader array of AI and HPC tasks.
- A new Transformer Engine that enables H100 to deliver up to 9x faster AI training and up to 30x faster AI inference speedups on large language models compared to the prior generation A100.
- The new NVLink Network interconnect. This enables GPU-to-GPU communication among up to 256 GPUs across multiple compute nodes
- Secure MIG partitions the GPU into isolated, right-size instances to maximize QoS (quality of service) for smaller workloads. 

Please run the command `!nvidia-smi topo -m` in the cell below to display your node’s GPU and  NIC communication topology.


In [ ]:
!srun --partition=primary -n1 --gres=gpu:8 nvidia-smi topo -m

**Expected Output**:

<center><img src="images/dgx-h100-terminal-topo.png" width="850px" height="750px" alt-text="h100-topo-terminal"/></center>


GPU0 to GPU7 are connected via NVLink (NV18). Each GPU in H100 is connected to all other GPUs via the fourth-generation NVLink. As a result, this increases the GPU-to-GPU direct bandwidth to 900 gigabytes per second (GB/s) compared to A100. You can also test the command `nvidia-smi topo -mp` to display the GPUDirect communication matrix via only PCI for the system.
The diagram below shows the high-level topology overview of DGX H100. Within a node, each of the 8 GPUs can efficiently communicate with each other via NVLink Switch (NVSwitch).


<center><img src="images/dgxh100-topo.png" width="700px" height="700px" alt-text="Arc"/></center>

 


 #### H100 SM Architecture 

Building upon the NVIDIA A100 Tensor Core GPU SM architecture, the H100 SM quadruples A100’s peak per-SM floating point computational power due to the introduction of FP8. It doubles A100’s raw SM computational power on all previous Tensor Core and FP32 / FP64 data types and clock-for-clock. Hopper’s new fourth-generation Tensor Core, Tensor Memory Accelerator, new SM, and general H100 architecture improvements often deliver up to 3x faster HPC and AI performance.   

<center>NVIDIA H100 Tensor Core GPU Performance Specs</center>
<center><img src="images/H100-tensor-core-gpu-performance-spec.png" width="500px" height="500px" alt-text="Arc"/></center>

Key SM features on Fourth-generation Tensor Cores include:

- Up to 6x faster chip-to-chip compared to A100, including per-SM speedup, additional SM count, and higher clocks of H100.
- On a per SM basis, the Tensor Cores deliver 2x the MMA (Matrix Multiply Accumulate) computational rates of the A100 SM on equivalent data types and 4x the rate of A100 using the new FP8 data type. 
- Sparsity feature exploits fine-grained structured sparsity in deep learning networks, doubling the performance of standard Tensor Core operations.


#### Hopper FP8 Data Format

The H100 GPU adds FP8 Tensor Cores to accelerate AI training and inference. As shown in the diagram below, FP8 Tensor Cores support FP32 and FP16 accumulators, and two new FP8 input types:

- E4M3 with 4 exponent bits, 3 mantissa bits, and 1 sign bit
- E5M2, with 5 exponent bits, 2 mantissa bits, and 1 sign bit.

E4M3 supports computations requiring less dynamic range with more precision, while E5M2 provides a wider dynamic range and less precision. The recommended use of FP8 encodings is E4M3 for weight and activation tensors, and E5M2 for gradient tensors. FP8, compared to FP16 or BF16, halves data storage requirements and doubles throughput. 
New Hopper FP8 Precisions - 2x throughput and half the footprint of H100 FP16 / BF16. H100 FP8 Tensor Core 6x throughput compared to A100 FP16 Tensor Core. 


<center><img src="images/H100-FP8-precision-1.png" width="500px" height="500px" alt-text="Arc"/></center>


## NVIDIA Collective Communication Library (NCCL)

The [NVIDIA Collective Communications Library](https://developer.nvidia.com/nccl) provides inter-GPU communication primitives that are topology-aware and can be easily integrated into applications. NCCL implements both collective communication and point-to-point send/receive primitives. It is not a full-blown parallel programming framework but a library that accelerates inter-GPU communication. NCCL provides the following collective communication primitives:

- **AllReduce**: The AllReduce operation performs reductions on data (for example, sum, min, max) across devices and stores the result in the receive buffer of every rank. In a sum `allreduce` operation between k ranks, each rank will provide an array in of N values and receive identical results in an array out of N values, where `out[i] = in0[i]+in1[i]+…+in(k-1)[i]`.

<center><img src="images/allreduce.png" width="380px" height="380px" alt-text="allreduce"/> </center>


- **Broadcast**: The Broadcast operation copies an N-element buffer from the root rank to all the ranks.

<center><img src="images/broadcast.png" width="380px" height="380px" alt-text="broadcast"/> </center>

- **Reduce**: The Reduce operation performs the same operation as `AllReduce`, but stores the result only in the receive buffer of a specified root rank. A Reduce, followed by a Broadcast, is equivalent to the AllReduce operation.

  <center><img src="images/reduce.png" width="380px" height="380px" alt-text="reduce"/> </center>

- **AllGather**: The AllGather operation gathers N values from k ranks into an output buffer of size `k*N` and distributes that result to all ranks. The output is ordered by the rank index. The AllGather operation is, therefore, impacted by a different rank to device mapping. Executing ReduceScatter, followed by AllGather, is equivalent to the AllReduce operation.

<center><img src="images/allgather.png" width="380px" height="380px" alt-text="allgather"/> </center>

- **ReduceScatter**: The ReduceScatter operation performs the same operation as Reduce, except that the result is scattered in equal-sized blocks between ranks, each rank getting a chunk of data based on its rank index. A different rank-to-device mapping impacts the ReduceScatter operation since the ranks determine the data layout.

  <center><img src="images/reducescatter.png" width="380px" height="380px" alt-text="reducescatter"/> </center>


Additionally, NCCL allows for point-to-point send/receive communication, which permits scatter, gather, or all-to-all operations.
NCCL conveniently removes the need for developers to optimize their applications for specific machines. It provides fast collectives over multiple GPUs both within and across nodes. It supports various interconnect technologies, including PCIe, NVLINK, InfiniBand Verbs, and IP sockets. NCCL has great application in Deep Learning Frameworks, where the AllReduce collective is heavily used for neural network training. Efficient scaling of neural network training is possible with the multi-GPU and multi-node communication provided by NCCL.


#### Environment Variables

NCCL has an extensive set of [environment variables](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/env.html) to tune for specific usage.  There are two categories of environment variables: system configuration and Debugging.

Examples of system configuration are:
- **NCCL_DEBUG**: This variable controls the debug information that is displayed from NCCL. This variable is commonly used for debugging, using the following acceptable values:

```text
VERSION - Prints the NCCL version at the start of the program.
WARN    - Prints an explicit error message whenever any NCCL call errors out.
INFO    - Prints debug information
TRACE   - Prints replayable trace information on every call.
e.g. NCCL_DEBUG=WARN
```   
- **NCCL_SOCKET_IFNAME**: The variable specifies which IP interfaces to use for communication. A list of prefixes used to filter NCCL interfaces must be defined to specify acceptable values. This includes prefixes, such as `, symbol` for multiple prefixes, `^ symbol`, to exclude interfaces starting with any prefix in a list, `= character` to match (or not) an exact interface name. Examples are given below:
  
```text
eth        : Use all interfaces starting with eth, e.g. eth0, eth1, …
=eth0      : Use only interface eth0
=eth0,eth1 : Use only interfaces eth0 and eth1
^docker    : Do not use any interface starting with docker
^=docker0  : Do not use interface docker0.

e.g. NCCL_SOCKET_IFNAME==ens1f0
```
- **NCCL_IB_DISABLE**: The variable prevents NCCL from using IB/RoCE transport. Instead, NCCL will fall back to using IP sockets. When the value is set to 1, InfiniBand Verbs are disabled for communication (and another method, e.g., IP sockets, is used). For example: `NCCL_IB_DISABLE=1`

- **NCCL_ALGO**: The variable defines which algorithms NCCL will use. A table of accepted algorithm include:
 |Version|Algorithm| 
 |--|--|
 |2.5+ | Ring|
 |2.5+ | Tree|
 | 2.5 to 2.13| Collnet|
 | 2.14+ |CollnetChain|
 |2.14+ |CollnetDirect|
 | 2.17+ | NVLS|
 | 2.18+ | NVLSTree|
 | 2.23+ | PAT|


`NVLS` and `NVLSTree` enable NVLink SHARP offload. For example, `NCCL_ALGO="ring,collnetdirect;allreduce:tree,collnetdirect;broadcast:ring"` Will enable ring and collnetdirect for all functions, then enable tree and collnetdirect for allreduce and ring for broadcast. You can also set the variabl as `NCCL_ALGO="ring`.


From the debugging category, let’s look at the environment variable for P2P:

- **NCCL_P2P_DISABLE**: The NCCL_P2P_DISABLE variable disables the peer-to-peer (P2P) transport, which uses CUDA direct access between GPUs via NVLink or PCI. The value should be set to 1 to disable direct GPU-to-GPU (P2P) communication.
- **NCCL_P2P_LEVEL**: The NCCL_P2P_LEVEL variable allows the user to finely control when to use the peer-to-peer (P2P) transport between GPUs. The level defines the maximum distance between GPUs where NCCL will use the P2P transport. A short string representing the path type should be used to specify the topographical cutoff for using the P2P transport. If this isn’t specified, NCCL will attempt to optimally select a value based on the architecture and environment in which it’s run.


**Acceptable values and legacy integers**
```text
LOC or 0: Never use P2P (always disabled)
NVL : Use P2P when GPUs are connected through NVLink
PIX or 1: Use P2P when GPUs are on the same PCI switch.
PXB or 2: Use P2P when GPUs are connected through PCI switches (potentially multiple hops).
PHB or 3: Use P2P when GPUs are on the same NUMA node. Traffic will go through the CPU.
SYS or 4: Use P2P between NUMA nodes, potentially crossing the SMP interconnect (e.g. QPI/UPI).
```




### NVLink & NVLink Switch

NVLink is a 3.6 TB/s bidirectional, direct GPU-to-GPU interconnect that scales multi-GPU input and output (IO) within a server. The NVIDIA NVLink Switch chips connect multiple NVLinks to provide all-to-all GPU communication at full NVLink speed across the entire rack. To enable high-speed, collective operations, each NVLink Switch has engines for NVIDIA Scalable Hierarchical Aggregation and Reduction Protocol (SHARP)™ for in-network reductions and multicast acceleration.

 <center><img src="images/nvlink.png" width="850px" height="650px" alt-text="nvlink"/> </center>
 <center><img src="images/nvlink-nvswitch.png" width="850px" height="650px" alt-text="nvlink"/> </center>
<br>


Having learned sysem topology, we will proceed to apply the knowledge about learning distributed training strategy. Please click on the `Next Notebook` link below to proceed to the data parallelism lab.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](data-parallelism.ipynb) </b> </div></center>


---
## References

- https://www.nvidia.com/en-us/data-center/nvlink/
- https://developer.nvidia.com/blog/nvidia-nvlink-and-nvidia-nvswitch-supercharge-large-language-model-inference
- https://images.nvidia.com/aem-dam/en-zz/Solutions/data-center/nvidia-ampere-architecture-whitepaper.pdf
- https://medium.com/red-buffer/getting-started-with-pytorch-distributed-54ae933bb9f0
- https://developer.nvidia.com/nsight-systems/get-started
- FP8 Format for deep learning (https://arxiv.org/pdf/2209.05433)
- https://www.nvidia.com/en-us/data-center/nvlink/

## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.